# Query Pipeline
> Chain multiple query transformers into a sequential pipeline.

`QueryTransformPipeline` runs a sequence of transformers in order.
This notebook also covers `ContextualQueryTransformer` and
`ConversationRewriter` for adding context and rewriting queries
from conversation history.

**Time:** ~8 minutes

## Setup

In [ ]:
from anchor.query import (
    QueryTransformPipeline,
    ContextualQueryTransformer,
    ConversationRewriter,
)
from anchor.models import QueryBundle

## ContextualQueryTransformer
Adds surrounding context (e.g., system prompt, user profile) to the query
before retrieval.

In [ ]:
context_fn = lambda q: QueryBundle(
    query_str=f"Given the context: {q.query_str}"
)

contextual = ContextualQueryTransformer(context_fn=context_fn)

query = QueryBundle(query_str="What is context engineering?")
result = contextual.transform(query)

print(f"Original:    {query.query_str}")
print(f"Contextual:  {result[0].query_str}")

## ConversationRewriter
Rewrites a follow-up query using conversation history to make it
self-contained for retrieval.

In [ ]:
rewrite_fn = lambda q: QueryBundle(
    query_str=f"Rewritten: {q.query_str}"
)

rewriter = ConversationRewriter(rewrite_fn=rewrite_fn)

follow_up = QueryBundle(query_str="How does it handle memory?")
result = rewriter.transform(follow_up)

print(f"Follow-up:   {follow_up.query_str}")
print(f"Rewritten:   {result[0].query_str}")

## Realistic Conversation Rewriting
In practice, the rewriter uses conversation history to resolve pronouns
and implicit references.

In [ ]:
# Simulate conversation-aware rewriting
conversation_history = [
    {"role": "user", "content": "Tell me about Anchor's memory module"},
    {"role": "assistant", "content": "The memory module manages conversation..."},
]

def history_aware_rewrite(q: QueryBundle) -> QueryBundle:
    """Rewrite using conversation history for context."""
    last_topic = conversation_history[0]["content"]
    return QueryBundle(
        query_str=f"Regarding {last_topic}: {q.query_str}"
    )

smart_rewriter = ConversationRewriter(rewrite_fn=history_aware_rewrite)

# "it" refers to the memory module from conversation history
ambiguous = QueryBundle(query_str="How does it handle eviction?")
resolved = smart_rewriter.transform(ambiguous)

print(f"Ambiguous: {ambiguous.query_str}")
print(f"Resolved:  {resolved[0].query_str}")

## Build a Query Pipeline
`QueryTransformPipeline` chains transformers sequentially. The output
of each transformer feeds into the next.

In [ ]:
pipeline = QueryTransformPipeline(
    transformers=[contextual, rewriter]
)

print(f"Pipeline: {type(pipeline).__name__}")
print(f"Stages:   {len(pipeline.transformers)}")
for i, t in enumerate(pipeline.transformers):
    print(f"  [{i}] {type(t).__name__}")

## Run the Pipeline
`transform()` passes the query through each stage in order.

In [ ]:
query = QueryBundle(query_str="What is context engineering?")
print(f"Input: {query.query_str}\n")

result = pipeline.transform(query)

print(f"Pipeline produced {len(result)} queries:")
for i, q in enumerate(result):
    print(f"  [{i}] {q.query_str}")

## Multi-Stage Pipeline Example
Combine context injection, rewriting, and expansion in a single
pipeline.

In [ ]:
from anchor.query import HyDETransformer

# Stage 1: Add context
add_context = ContextualQueryTransformer(
    context_fn=lambda q: QueryBundle(
        query_str=f"[User: developer] {q.query_str}"
    )
)

# Stage 2: Rewrite for clarity
clarify = ConversationRewriter(
    rewrite_fn=lambda q: QueryBundle(
        query_str=q.query_str.replace("it", "Anchor framework")
    )
)

# Stage 3: HyDE expansion
hyde = HyDETransformer(
    generate_fn=lambda q: f"A document about {q} would cover..."
)

full_pipeline = QueryTransformPipeline(
    transformers=[add_context, clarify, hyde]
)

print(f"Full pipeline stages:")
for i, t in enumerate(full_pipeline.transformers):
    print(f"  [{i}] {type(t).__name__}")

In [ ]:
# Run the full pipeline
query = QueryBundle(query_str="How does it handle memory?")
print(f"Input: {query.query_str}\n")

results = full_pipeline.transform(query)

print(f"Pipeline output ({len(results)} queries):")
for i, q in enumerate(results):
    label = q.query_str[:70]
    suffix = "..." if len(q.query_str) > 70 else ""
    print(f"  [{i}] {label}{suffix}")

## Pipeline Inspection
Track how the query transforms through each stage.

In [ ]:
# Step through the pipeline manually to see each transformation
query = QueryBundle(query_str="How does it handle memory?")
print(f"Stage 0 (input): {query.query_str}\n")

current = [query]
for i, transformer in enumerate(full_pipeline.transformers):
    stage_name = type(transformer).__name__
    # Apply transformer to first query in current list
    current = transformer.transform(current[0])
    print(f"Stage {i + 1} ({stage_name}):")
    for q in current:
        print(f"  -> {q.query_str[:70]}")
    print()

## Key Takeaways
- `QueryTransformPipeline` chains transformers in sequence
- `ContextualQueryTransformer` injects external context into queries
- `ConversationRewriter` resolves ambiguity from conversation history
- Pipelines enable composable query processing workflows
- Step through the pipeline manually to debug transformations
- Combine context, rewriting, and expansion for production-grade query handling

**Back to:** [Query README](README.md)